In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/processed/train.csv")

# Missingness Analysis

In [ ]:
missing = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_pct": (
        train.isna().mean() * 100
    ).round(2)
})

missing.sort_values(
    by="missing_pct",
    ascending=False
)

Missingness Analysis

MonthlyIncome contains approximately 20% missing values.

NumberOfDependents contains approximately 2.5% missing values.

All remaining variables contain no missing values.

# Information Value (IV)

In [ ]:
from optbinning import OptimalBinning

In [ ]:
def calculate_iv(df, variable, target):

    x = df[variable]
    y = df[target]

    optb = OptimalBinning(
        name=variable,
        dtype="numerical"
    )

    optb.fit(x, y)

    return optb.binning_table.build()["IV"].sum()

In [ ]:
candidate_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]

In [ ]:
iv_results = []

for variable in candidate_variables:

    iv = calculate_iv(
        train,
        variable,
        "SeriousDlqin2yrs"
    )

    iv_results.append(
        {
            "variable": variable,
            "IV": iv
        }
    )

iv_table = pd.DataFrame(iv_results)

iv_table.sort_values(
    by="IV",
    ascending=False
)

# Predictive Power

In [ ]:
def iv_category(iv):

    if iv < 0.02:
        return "Useless"

    elif iv < 0.1:
        return "Weak"

    elif iv < 0.3:
        return "Medium"

    elif iv < 0.5:
        return "Strong"

    else:
        return "Very Strong"

In [ ]:
iv_table["strength"] = (
    iv_table["IV"]
    .apply(iv_category)
)

iv_table.sort_values(
    by="IV",
    ascending=False
)

### Business Relevance Assessment

Retain:

- Age
- Delinquency variables
- Revolving Utilization
- Monthly Income

Review:

- Debt Ratio
- Number of Dependents

### Multicollinearity Review

Three delinquency-related variables exhibit extremely high pairwise correlations (>0.98):

- NumberOfTimes90DaysLate
- NumberOfTime30-59DaysPastDueNotWorse
- NumberOfTime60-89DaysPastDueNotWorse

Given the strong overlap in information content, it is unlikely that all three variables will remain in the final scorecard.

At this stage all variables are retained for further WoE, binning and logistic regression analysis.

A final decision will be made during Feature Selection and Model Development.

### Information Value Analysis

The strongest predictors were:

- RevolvingUtilizationOfUnsecuredLines (IV = 2.22)
- NumberOfTimes90DaysLate (IV = 1.70)
- NumberOfTime30-59DaysPastDueNotWorse (IV = 1.46)
- NumberOfTime60-89DaysPastDueNotWorse (IV = 1.16)
- Age (IV = 0.51)

The delinquency-related variables demonstrated exceptionally high predictive power, which is consistent with their strong relationship to future credit deterioration.

Several variables exhibited moderate predictive power:

- NumberOfOpenCreditLinesAndLoans
- DebtRatio
- MonthlyIncome
- NumberRealEstateLoansOrLines

NumberOfDependents showed weak predictive power but was retained for further evaluation.

No variables were excluded at this stage.